# Density of States Effective Mass with VASP

The DOS effective mass is necessary for the final PV Figure of Merit, and can be calculated from the results of a dense DOS calculation centered on a band-edge k-point.

#### Pre-requisites:
* Structural Relaxation
* Band Structure (to choose the central point)

1) Import the necessary libraries

In [1]:
import numpy as np

import solphin.band_structure as band_structure
import solphin.dos as dos
import solphin.vasp_inputs as vasp_inputs

2) Define our computational regime. This should be the same regime used to relax the structure.

In [2]:
functional = 'HSE06'  # The DFT functional
encut = 450           # Plane-wave basis set energy cutoff
kspacing = 0.2        # k-point sampling density

3) Load the optimised atomic structure from a previous relaxation. Here we read from the bundled test data. 

In [3]:
structure = vasp_inputs.read_structure_pmg("../tests/data/Cu2GeS3/Relax/CONTCAR")

4. Ensure the structure has the same canonical orientation used for the band structure.

In [4]:
canonical_structure, _ = band_structure.generate_band_structure_path(structure=structure, definition="bradcrack")

Generated high-symmetry path of 239 k-points


5) Write the VASP input files. Here we write to `workdir/optics` which is untracked, and can be used to try out these tutorials. This requires having set up your VASP `POTCAR`s with `pymatgen`.

* We choose the gamma point (0,0,0) as the location of our band-edge.

In [5]:
dos.write_eff_mass(
    k0_frac = np.array([0,0,0]),
    structure = canonical_structure,
    functional = functional,
    encut = encut,
    folder = "workdir/eff_mass"
)

6) Run the VASP calculation. The specifics will depend on your particular machine, but will require invoking the `vasp_std` command in a suitable environment.

7) Fit the effective masses from the calculation results. We point again to the reference data.

In [6]:
result = dos.compute_dos(filepath=f"../tests/data/Cu2GeS3/DOS_HDFT/vasprun.xml", code="vasp")
print(result)

  Computing electron FOM DOS effective mass...
  Computing hole FOM DOS effective mass...

  DOS Result Summary
  Primary carrier     : Electrons
  Band edge (CBM)     : 1.402 eV
  Cell volume         : 2.234e-28 m³

  ── FOM DOS Effective Masses ───────────────────────
  Electrons (fitted at CBM: 1.402 eV)
  DOS effective mass    : 0.103528 mₑ  (9.431e-32 kg)  ← primary carrier
  Fit quality           : 0.640706 R²
  Points fitted         : 11
  Energy window         : 0.1500 eV


  Holes (fitted at VBM: 0.000 eV)
  DOS effective mass    : 1.080489 mₕ  (9.843e-31 kg)
  Fit quality           : 0.627290 R²
  Points fitted         : 11
  Energy window         : 0.1500 eV


  Γₚᵥ DOS mass √(mₑm_h) : 0.334456 m₀

